# 00 — Shared protocol

อ่าน notebook นี้ก่อน R0–R6 เพื่อเข้าใจ candidate blocking, entity-aware split, metrics และ evaluation harness ที่ทุก experiment ใช้ร่วมกัน.

In [ ]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import Markdown, display

def find_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents,
                  Path(r'D:/66070260-Year3_Term2/Project1/Code')]
    for candidate in candidates:
        if (candidate / 'exp_lib.py').exists(): return candidate
    raise FileNotFoundError('Project root containing exp_lib.py was not found')

ROOT = find_root(); EXP = ROOT / 'experiments'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

def source(module, *names):
    for name in names:
        display(Markdown(f'### `{module.__name__}.{name}`'))
        print(inspect.getsource(getattr(module, name)))

def read_json(relative):
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))

print('Project root:', ROOT)


## 1. Candidate pairs และ fixed errors

ระบบสร้าง candidate pairs ก่อน score. คู่จริงที่ไม่เคยผ่าน blocking จะไม่มี probability ให้ model แก้ได้ จึงถูกนับเป็น fixed FN ใน harness.

In [ ]:
import exp_lib
source(exp_lib, 'build_cache', 'split_constants', 'evaluate')
cache = exp_lib.build_cache(); consts = exp_lib.split_constants(cache)
print('Scored cache:', f'{len(cache):,} rows'); print('Test constants:', consts['test'])

## 2. Nested entity-aware split

แบ่งตาม entity/person ไม่ใช่แบ่งทีละ pair: `model_train → model_calibration → ga_validation → sealed test`. คู่คร่อม role ถูก drop เพื่อกัน leakage.

In [ ]:
source(exp_lib, 'build_nested_entity_split', 'nested_split_constants')
assigned, manifest = exp_lib.build_nested_entity_split(cache, seed=42)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

## 3. Metrics

`cost = 5×FP + 1×FN + 0.02×REVIEW`; การรวมคนผิดมีน้ำหนักสูงกว่า missed match. REVIEW ไม่ใช่ error แต่เป็นภาระงานคนตรวจ.